## SelfGNN — Synthetic Feature Extraction (Uniform 4+4 Schema)

**Prerequisite:** Run `preprocess_synthetic.ipynb` first to generate `user2id.pkl`, `merchant2id.pkl`.

**Uniform schema (identical across all datasets):**

| Side | Dim | Features |
|------|-----|----------|
| User | 4 | `interaction_count`, `avg_interaction_value`, `unique_merchant_count`, `activity_span_days` |
| Merchant | 4 | `interaction_count`, `avg_interaction_value`, `unique_user_count`, `category_id` |
| Edge | 1 | `normalized_interaction_value` (sigmoid(log1p(amount) / p75)) |

**Outputs:** `Datasets/synthetic-merchant/`
- `user_features.npy` — shape (N_users, 4), float32, log1p+MinMax
- `merchant_features.npy` — shape (N_merchants, 4), float32
- `edge_weights.pkl` — `{(uid, mid): avg_normalized_amount}`
- `feature_meta.json` — column index metadata
- `bipartite_graph_sample.png` — geometric graph visualization

In [1]:
# ── Configuration ──────────────────────────────────────────────────────────────
OUT_DIR      = '../Datasets/synthetic-merchant/'
RAW_PATH     = '../datasetRaw/synthetic/dataset.csv'
DATASET_NAME = 'synthetic-merchant'

import os, json, pickle
import numpy as np
import pandas as pd
from collections import defaultdict

with open(OUT_DIR + 'user2id.pkl', 'rb') as f:
    user2id = pickle.load(f)
with open(OUT_DIR + 'merchant2id.pkl', 'rb') as f:
    merchant2id = pickle.load(f)

usrnum      = len(user2id)
merchantnum = len(merchant2id)
print(f'Loaded mappings: {usrnum:,} users, {merchantnum:,} merchants')

Loaded mappings: 81,433 users, 48 merchants


In [2]:
# ── Cell 2: Compute p75 of log-amount for normalization ────────────────────────
sample      = pd.read_csv(RAW_PATH, nrows=5)
has_mcc     = 'merchant_category_code' in sample.columns
use_cols    = ['customer_id', 'merchant_name', 'timestamp', 'amount_mnt']
if has_mcc:
    use_cols.append('merchant_category_code')
print(f'Columns: {list(sample.columns)}')
print(f'Has MCC: {has_mcc}')

all_amounts = []
for chunk in pd.read_csv(RAW_PATH, chunksize=500_000, low_memory=False, usecols=use_cols):
    chunk['customer_id']   = chunk['customer_id'].astype(str)
    chunk['merchant_name'] = chunk['merchant_name'].astype(str)
    chunk = chunk[
        chunk['customer_id'].isin(user2id) &
        chunk['merchant_name'].isin(merchant2id)
    ].dropna(subset=['amount_mnt'])
    all_amounts.append(chunk['amount_mnt'].clip(lower=0).values)

amounts_flat = np.concatenate(all_amounts)
log_amounts  = np.log1p(amounts_flat)
p75          = float(max(np.percentile(log_amounts, 75), 1.0))
print(f'Amount p75 (log scale): {p75:.4f}')

def norm_amount(amt: np.ndarray) -> np.ndarray:
    """sigmoid(log1p(amount) / p75) → (0, 1)"""
    return 1.0 / (1.0 + np.exp(-np.log1p(np.clip(amt, 0, None)) / p75))

Columns: ['transaction_id', 'account_id', 'customer_id', 'timestamp', 'amount_mnt', 'balance_before_mnt', 'balance_after_mnt', 'transaction_type', 'channel', 'merchant_category_code', 'merchant_name', 'device_id']
Has MCC: True
Amount p75 (log scale): 12.0388


In [3]:
# ── Cell 3: Build per-user/merchant/edge statistics ────────────────────────────
u_count      = np.zeros(usrnum,      dtype=np.int64)
u_val_sum    = np.zeros(usrnum,      dtype=np.float64)
u_merchants  = [set() for _ in range(usrnum)]
u_min_ts     = np.full(usrnum,  np.inf)
u_max_ts     = np.full(usrnum, -np.inf)

m_count      = np.zeros(merchantnum, dtype=np.int64)
m_val_sum    = np.zeros(merchantnum, dtype=np.float64)
m_users      = [set() for _ in range(merchantnum)]
m_mcc: dict  = {}

edge_val_sum: dict = defaultdict(float)
edge_cnt:     dict = defaultdict(int)

for chunk in pd.read_csv(RAW_PATH, chunksize=500_000, low_memory=False, usecols=use_cols):
    chunk['customer_id']   = chunk['customer_id'].astype(str)
    chunk['merchant_name'] = chunk['merchant_name'].astype(str)
    chunk['ts']            = pd.to_datetime(chunk['timestamp'], errors='coerce')
    chunk = chunk[
        chunk['customer_id'].isin(user2id) &
        chunk['merchant_name'].isin(merchant2id)
    ].dropna(subset=['amount_mnt', 'ts'])

    chunk['uid']      = chunk['customer_id'].map(user2id)
    chunk['mid']      = chunk['merchant_name'].map(merchant2id)
    chunk['norm_val'] = norm_amount(chunk['amount_mnt'].values)
    chunk['unix_ts']  = chunk['ts'].astype(np.int64) // 10**9

    iter_cols = ['uid', 'mid', 'norm_val', 'unix_ts']
    if has_mcc:
        iter_cols.append('merchant_category_code')

    for row in chunk[iter_cols].itertuples(index=False):
        uid, mid = int(row.uid), int(row.mid)
        v = float(row.norm_val)
        ts = int(row.unix_ts)

        u_count[uid]   += 1
        u_val_sum[uid]  += v
        u_merchants[uid].add(mid)
        if ts < u_min_ts[uid]: u_min_ts[uid] = ts
        if ts > u_max_ts[uid]: u_max_ts[uid] = ts

        m_count[mid]   += 1
        m_val_sum[mid]  += v
        m_users[mid].add(uid)

        if has_mcc and mid not in m_mcc:
            mcc_val = getattr(row, 'merchant_category_code', None)
            if pd.notna(mcc_val):
                m_mcc[mid] = int(mcc_val)

        edge_val_sum[(uid, mid)] += v
        edge_cnt[(uid, mid)]     += 1

print(f'Total events: {u_count.sum():,}')

Total events: 1,944,655


In [4]:
# ── Cell 4: Build feature arrays ───────────────────────────────────────────────
u_avg_val      = np.where(u_count > 0, u_val_sum / u_count, 0.0).astype(np.float32)
u_unique_merch = np.array([len(s) for s in u_merchants], dtype=np.float32)
u_span_days    = np.where(
    np.isfinite(u_min_ts) & np.isfinite(u_max_ts),
    (u_max_ts - u_min_ts) / 86400.0, 0.0,
).astype(np.float32)

m_avg_val      = np.where(m_count > 0, m_val_sum / m_count, 0.0).astype(np.float32)
m_unique_users = np.array([len(s) for s in m_users], dtype=np.float32)

unique_mccs = sorted(set(m_mcc.values())) if m_mcc else [0]
mcc2idx     = {v: i for i, v in enumerate(unique_mccs)}
m_cat       = np.array(
    [float(mcc2idx.get(m_mcc.get(mid, 0), 0)) for mid in range(merchantnum)],
    dtype=np.float32,
)

user_features = np.stack([
    u_count.astype(np.float32),  # 0: interaction_count
    u_avg_val,                    # 1: avg_interaction_value
    u_unique_merch,               # 2: unique_merchant_count
    u_span_days,                  # 3: activity_span_days
], axis=1)

merchant_features = np.stack([
    m_count.astype(np.float32),  # 0: interaction_count
    m_avg_val,                    # 1: avg_interaction_value
    m_unique_users,               # 2: unique_user_count
    m_cat,                        # 3: category_id (MCC index)
], axis=1)

print(f'User features     : {user_features.shape}')
print(f'Merchant features : {merchant_features.shape}')

User features     : (81433, 4)
Merchant features : (48, 4)


In [5]:
# ── Cell 5: Normalize features ─────────────────────────────────────────────────
def normalize_features(feat: np.ndarray) -> np.ndarray:
    out = feat.copy()
    for col in [0, 2, 3]:
        vals = np.log1p(feat[:, col].clip(min=0))
        mn, mx = vals.min(), vals.max()
        out[:, col] = ((vals - mn) / max(mx - mn, 1e-8)).astype(np.float32)
    return out

user_features_norm     = normalize_features(user_features)
merchant_features_norm = normalize_features(merchant_features)

print('After normalization:')
print(f'  User   min/max per col: {user_features_norm.min(axis=0)} / {user_features_norm.max(axis=0)}')
print(f'  Merch  min/max per col: {merchant_features_norm.min(axis=0)} / {merchant_features_norm.max(axis=0)}')

After normalization:
  User   min/max per col: [0.        0.6579662 0.        0.       ] / [1.        0.7603593 1.        1.       ]
  Merch  min/max per col: [0.         0.70939225 0.         0.        ] / [1.        0.7101949 1.        1.       ]


In [6]:
# ── Cell 6: Edge weights dict ──────────────────────────────────────────────────
edge_weights = {
    (uid, mid): float(edge_val_sum[(uid, mid)] / edge_cnt[(uid, mid)])
    for (uid, mid) in edge_cnt
}
vals = np.array(list(edge_weights.values()))
print(f'Edge weights: {len(edge_weights):,} pairs')
print(f'  min={vals.min():.4f}, max={vals.max():.4f}, mean={vals.mean():.4f}')

Edge weights: 946,666 pairs
  min=0.6220, max=0.8008, mean=0.7099


In [7]:
# ── Cell 7: Bipartite graph visualization ──────────────────────────────────────
# Geometric bipartite layout — users (left, blue circles) ↔ merchants (right, orange squares).
# Nodes are the top-N highest-degree users and their connected merchants.
# Edge thickness ∝ interaction weight (normalized to [0,1]).

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from collections import defaultdict as _dd

    np.random.seed(42)
    N_USR_SHOW = 20
    N_MRC_SHOW = 25

    # Index edge_weights by user
    u_to_m: dict = _dd(list)
    for (uid, mid), w in edge_weights.items():
        u_to_m[uid].append((mid, w))

    # Select highest-degree users
    top_users  = sorted(u_to_m.keys(), key=lambda u: len(u_to_m[u]), reverse=True)
    sample_u   = top_users[:N_USR_SHOW]

    # Gather connected merchants, capped at N_MRC_SHOW
    sample_m_set: set = set()
    for uid in sample_u:
        for mid, _ in u_to_m[uid]:
            sample_m_set.add(mid)
    sample_m    = sorted(sample_m_set)[:N_MRC_SHOW]
    m_visible   = set(sample_m)

    n_u, n_m = len(sample_u), len(sample_m)

    # Bipartite layout: users x=0, merchants x=1
    u_pos = {uid: (0.0, i / max(n_u - 1, 1)) for i, uid in enumerate(sample_u)}
    m_pos = {mid: (1.0, i / max(n_m - 1, 1)) for i, mid in enumerate(sample_m)}

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.axis('off')
    ax.set_title(
        f'Bipartite Interaction Graph — {DATASET_NAME}\n'
        f'(top-{n_u} users by degree  ·  {n_m} connected merchants)',
        fontsize=13, fontweight='bold', pad=14,
    )

    visible_ws = [w for uid in sample_u for mid, w in u_to_m[uid] if mid in m_visible]
    w_max = max(visible_ws) if visible_ws else 1.0

    # Draw edges
    for uid in sample_u:
        for mid, w in u_to_m[uid]:
            if mid not in m_visible:
                continue
            p1, p2 = u_pos[uid], m_pos[mid]
            ax.plot([p1[0], p2[0]], [p1[1], p2[1]],
                    color='#90A4AE', alpha=0.4,
                    linewidth=0.4 + 2.2 * (w / w_max), zorder=1)

    # User nodes (blue circles)
    ax.scatter([u_pos[u][0] for u in sample_u], [u_pos[u][1] for u in sample_u],
               s=220, c='#1565C0', zorder=4, edgecolors='white', linewidths=1.8, label='User')
    for uid, (x, y) in u_pos.items():
        ax.text(x - 0.05, y, f'u{uid}', ha='right', va='center', fontsize=7, color='#1565C0')

    # Merchant nodes (orange squares)
    ax.scatter([m_pos[m][0] for m in sample_m], [m_pos[m][1] for m in sample_m],
               s=220, c='#E65100', marker='s', zorder=4, edgecolors='white', linewidths=1.8, label='Merchant')
    for mid, (x, y) in m_pos.items():
        ax.text(x + 0.05, y, f'm{mid}', ha='left', va='center', fontsize=7, color='#E65100')

    ax.text(0.0, -0.08, f'Users  (n={n_u})',
            ha='center', fontsize=11, color='#1565C0', fontweight='bold',
            transform=ax.transData)
    ax.text(1.0, -0.08, f'Merchants  (n={n_m})',
            ha='center', fontsize=11, color='#E65100', fontweight='bold',
            transform=ax.transData)

    ax.set_xlim(-0.22, 1.22)
    ax.set_ylim(-0.12, 1.08)
    ax.legend(loc='upper center', ncol=2, fontsize=9, framealpha=0.8)

    plt.tight_layout()
    plt.savefig(OUT_DIR + 'bipartite_graph_sample.png', bbox_inches='tight', dpi=120)
    plt.show()
    print(f'Saved: bipartite_graph_sample.png  ({n_u} users, {n_m} merchants, {len(visible_ws)} edges shown)')

except Exception as e:
    print(f'Graph visualization skipped: {e}')

Saved: bipartite_graph_sample.png  (20 users, 25 merchants, 500 edges shown)


C:\Users\User\AppData\Local\Temp\ipykernel_19056\2872773007.py:85: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# ── Cell 8: Save all files + summary ───────────────────────────────────────────
np.save(OUT_DIR + 'user_features.npy',     user_features_norm)
np.save(OUT_DIR + 'merchant_features.npy', merchant_features_norm)

with open(OUT_DIR + 'edge_weights.pkl', 'wb') as f:
    pickle.dump(edge_weights, f)

meta = {
    'user_feature_names':     ['interaction_count', 'avg_interaction_value', 'unique_merchant_count', 'activity_span_days'],
    'merchant_feature_names': ['interaction_count', 'avg_interaction_value', 'unique_user_count', 'category_id'],
    'edge_feature_names':     ['normalized_interaction_value'],
    'user_value_col':         1,
    'merchant_value_col':     1,
    'edge_normalization':     'sigmoid(log1p(amount) / p75)',
    'dataset':                DATASET_NAME,
    'mcc_available':          has_mcc,
    'merchant_count_note':    f'Only {merchantnum} merchants — near-complete bipartite graph',
}
with open(OUT_DIR + 'feature_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Saved:')
for fname in ['user_features.npy', 'merchant_features.npy', 'edge_weights.pkl', 'feature_meta.json', 'bipartite_graph_sample.png']:
    path = OUT_DIR + fname
    size = os.path.getsize(path) if os.path.isfile(path) else None
    status = f'{size/1024:.0f} KB' if size else 'MISSING'
    print(f'  {fname:<35} {status}')

print()
print('=' * 60)
print(f'Feature Extraction Complete — {DATASET_NAME}')
print('=' * 60)
print(f'User feature shape     : {user_features_norm.shape}')
print(f'Merchant feature shape : {merchant_features_norm.shape}')
print(f'Edge weights           : {len(edge_weights):,} pairs')
print(f'Schema (uniform 4+4)   : user_dim={user_features_norm.shape[1]}, merch_dim={merchant_features_norm.shape[1]}')

Saved:
  user_features.npy                   1273 KB
  merchant_features.npy               1 KB
  edge_weights.pkl                    14973 KB
  feature_meta.json                   1 KB
  bipartite_graph_sample.png          751 KB

Feature Extraction Complete — synthetic-merchant
User feature shape     : (81433, 4)
Merchant feature shape : (48, 4)
Edge weights           : 946,666 pairs
Schema (uniform 4+4)   : user_dim=4, merch_dim=4
